In [ ]:
import os
os.environ["HUGGINGFACE_API"] = ""
os.environ["GIT_TOKEN"] = ""

In [17]:
import os
from huggingface_hub import login


huggingface_api = os.environ["HUGGINGFACE_API"]
git_token = os.environ["GIT_TOKEN"]

if huggingface_api is None:
    raise RuntimeError("❌ Missing HUGGINGFACE_API in .env")

login(token=huggingface_api)


In [3]:
!git clone https://{git_token}@github.com/BGKhanh/Reasoning-Techniques-on-LLM.git

Cloning into 'Reasoning-Techniques-on-LLM'...
remote: Enumerating objects: 2672, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 2672 (delta 125), reused 151 (delta 76), pack-reused 2456 (from 2)
Receiving objects: 100% (2672/2672), 4.63 MiB | 13.21 MiB/s, done.
Resolving deltas: 100% (1765/1765), done.


In [34]:
%cd /Reasoning-Techniques-on-LLM

/Reasoning-Techniques-on-LLM


In [5]:
!git pull

Already up to date.


In [6]:
# Add --ignore-installed if PyJWT error occurs or use "sudo apt remove python3-jwt"
import sys
!{sys.executable} -m pip install -Uqr requirements.txt --ignore-installed

In [7]:
!export FLASH_ATTENTION_SKIP_CUDA_BUILD=TRUE

In [8]:
import sys
!{sys.executable} -m pip install -q https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu130torch2.11-cp312-cp312-linux_x86_64.whl

In [43]:
%cd lm-evaluation-harness
import sys
!{sys.executable} -m pip install -qe .

[Errno 2] No such file or directory: 'lm-evaluation-harness'
/Reasoning-Techniques-on-LLM/lm-evaluation-harness


In [12]:
import os
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional

def _find_project_root(start: Path, max_up: int = 5) -> Path:
    cur = start.resolve()
    for _ in range(max_up):
        if (cur / "src" / "prompt_templates").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start  

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

VLLM_MODEL_ID = os.getenv("VLLM_MODEL_ID", "google/gemma-4-E4B-it-qat-w4a16-ct")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))

BASE_URL = os.getenv("BASE_URL", f"http://127.0.0.1:{VLLM_PORT}/v1")
base_url_completions = BASE_URL.rstrip("/") + "/completions"

OUTPUT_DIR = PROJECT_ROOT / "results" / "lm_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = /Reasoning-Techniques-on-LLM


# Host VLLM

In [27]:
# Khởi chạy vLLM OpenAI server cục bộ cho student LM (chạy sau cell Configuration).
# Tùy VRAM có thể thêm vào _vllm_cmd: --max-model-len 4096 --gpu-memory-utilization 0.9 --dtype bfloat16
import subprocess
import sys
import time
import urllib.error
import urllib.request

if "_vllm_proc" in globals() and _vllm_proc is not None and _vllm_proc.poll() is None:
    _vllm_proc.terminate()
    try:
        _vllm_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        _vllm_proc.kill()

_vllm_env = os.environ.copy()
_hf = (
    _vllm_env.get("HUGGINGFACE_API_KEY")
    or _vllm_env.get("HF_TOKEN")
    or _vllm_env.get("HUGGINGFACE_HUB_TOKEN")
    or _vllm_env.get("HUGGINGFACE_API", "")
)
if _hf:
    _vllm_env.setdefault("HF_TOKEN", _hf)
    _vllm_env.setdefault("HUGGINGFACE_HUB_TOKEN", _hf)

_vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", VLLM_MODEL_ID,
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--dtype", "bfloat16",
    "--gpu-memory-utilization", "0.975",
    # "--tensor-parallel-size", "2"
]

_vllm_log = OUTPUT_DIR / "vllm_server.log"
_flog = open(_vllm_log, "a", encoding="utf-8", buffering=1)
_flog.write(f"\n\n==== vLLM start {time.strftime('%Y-%m-%d %H:%M:%S')} ====\n")
_flog.write(" ".join(_vllm_cmd) + "\n")
_flog.flush()

print("Starting vLLM — log:", _vllm_log)
_vllm_proc = subprocess.Popen(
    _vllm_cmd,
    env=_vllm_env,
    stdout=_flog,
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
)

_health = BASE_URL.rstrip("/") + "/models"
_deadline = time.time() + float(os.getenv("VLLM_START_TIMEOUT_S", "1200"))
_last_err = None
while time.time() < _deadline:
    if _vllm_proc.poll() is not None:
        _flog.close()
        tail = _vllm_log.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError(
            f"vLLM exited early (code={_vllm_proc.returncode}). See {_vllm_log}. Tail:\n{tail}"
        )
    try:
        with urllib.request.urlopen(_health, timeout=5) as r:
            if r.status == 200:
                print(f"vLLM ready: {_health}")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        _last_err = e
    time.sleep(2.0)
else:
    if _vllm_proc.poll() is None:
        _vllm_proc.terminate()
    _flog.close()
    raise RuntimeError(f"vLLM did not become ready in time. Last error: {_last_err}. Log: {_vllm_log}")

Starting vLLM — log: /Reasoning-Techniques-on-LLM/results/lm_eval/vllm_server.log
vLLM ready: http://127.0.0.1:8000/v1/models


In [46]:
!PYTHONHASHSEED=42 \
CUBLAS_WORKSPACE_CONFIG=:4096:8 \
lm_eval \
  --model local-completions \
  --model_args model=model_name,base_url=http://127.0.0.1:8000/v1/models,num_concurrent=32 \
  --tasks vietnamese_ssa \
  --apply_chat_template \
  --log_samples \
  --output_path results/lm_eval/gemma_few_shot_3shot \
  --metadata '{"technique":"few_shot","language":"vi","n_shot":3}' \
  --confirm_run_unsafe_code

/bin/bash: line 1: lm_eval: command not found


# Host llama.cpp

In [ ]:
!pip install llama-cpp-python==0.3.30 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import subprocess
import time
import urllib.request
import urllib.error
import os
import atexit
from pathlib import Path

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ==========================================
MODEL_PATH = Path("gemma-3-4b-it-Q4_K_M.gguf") # Đảm bảo file này tồn tại
LLAMA_SERVER_BIN = "./llama-server"            # Đường dẫn tới file executable
PORT = 8080
LOG_FILE = Path("llama_server.log")

# Kiểm tra file thực thi
if not Path(LLAMA_SERVER_BIN).exists():
    raise FileNotFoundError(f"Không tìm thấy file chạy server tại: {LLAMA_SERVER_BIN}. Hãy đảm bảo bạn đã compile llama.cpp.")

# ==========================================
# 2. HÀM DỌN DẸP TIẾN TRÌNH TỰ ĐỘNG
# ==========================================
def cleanup_llama_server():
    """Hàm này sẽ tự động chạy để tắt server khi Kernel bị tắt hoặc restart."""
    global _llama_proc
    if '_llama_proc' in globals() and _llama_proc is not None:
        if _llama_proc.poll() is None: # Nếu process vẫn đang chạy
            print("\n[Cleanup] Đang tắt llama.cpp server an toàn...")
            _llama_proc.terminate()
            try:
                _llama_proc.wait(timeout=5)
                print("[Cleanup] Đã tắt server thành công.")
            except subprocess.TimeoutExpired:
                print("[Cleanup] Server không phản hồi, đang ép buộc tắt (kill)...")
                _llama_proc.kill()

# Đăng ký hàm dọn dẹp vào sự kiện thoát của Python
atexit.register(cleanup_llama_server)

# Dọn dẹp thủ công nếu chạy lại cell này nhiều lần
cleanup_llama_server()

# ==========================================
# 3. KHỞI CHẠY TIẾN TRÌNH (BACKGROUND PROCESS)
# ==========================================
llama_cmd = [
    LLAMA_SERVER_BIN,
    "-m", str(MODEL_PATH),
    "--port", str(PORT),
    "-c", "8192",         # Context window: 8192 tokens
    "-ngl", "99",         # Số layer offload lên GPU (99 = đẩy toàn bộ)
    "--host", "127.0.0.1",
    "--nobatch",          # (Tùy chọn) Giảm độ trễ cho từng request đơn lẻ
]

# Mở file log với buffering=1 (ghi trực tiếp từng dòng, không ngâm trong bộ nhớ đệm)
flog = open(LOG_FILE, "w", encoding="utf-8", buffering=1)
flog.write(f"==== Khởi chạy llama.cpp server lúc {time.strftime('%H:%M:%S')} ====\n")
flog.write(f"Lệnh: {' '.join(llama_cmd)}\n\n")

print(f"🚀 Đang khởi chạy llama.cpp server (Log: {LOG_FILE})...")

# Khởi tạo tiến trình ngầm
_llama_proc = subprocess.Popen(
    llama_cmd,
    stdout=flog,
    stderr=subprocess.STDOUT, # Gộp chung lỗi (stderr) vào file log
    env=os.environ.copy()     # Kế thừa biến môi trường hiện tại
)

# ==========================================
# 4. KIỂM TRA SỨC KHỎE (HEALTH CHECK & POLLING)
# ==========================================
health_url = f"http://127.0.0.1:{PORT}/health"
max_wait_seconds = 180 # GGUF lớn có thể mất thời gian load từ ổ cứng (đặc biệt là HDD)
deadline = time.time() + max_wait_seconds
last_err = None

print("⏳ Đang chờ mô hình nạp vào bộ nhớ (có thể mất vài phút)...")

while time.time() < deadline:
    # Kiểm tra xem tiến trình có bị crash giữa chừng không
    if _llama_proc.poll() is not None:
        flog.close()
        # Đọc 2000 ký tự cuối của file log để in ra nguyên nhân crash
        tail = LOG_FILE.read_text(encoding="utf-8", errors="replace")[-2000:]
        raise RuntimeError(f"❌ Server bị crash (Exit code: {_llama_proc.returncode}). Trích xuất log:\n{tail}")
    
    # Ping endpoint /health của llama.cpp
    try:
        with urllib.request.urlopen(health_url, timeout=2) as r:
            if r.status == 200:
                print(f"✅ llama.cpp đã tải xong và sẵn sàng nhận request tại: http://127.0.0.1:{PORT}")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        last_err = e # Lưu lại lỗi để báo cáo nếu hết thời gian
        pass
    
    time.sleep(2.0) # Đợi 2 giây trước khi ping lại
else:
    # Vòng lặp kết thúc mà không break -> Hết thời gian chờ
    cleanup_llama_server()
    flog.close()
    raise RuntimeError(f"⏳ Quá thời gian chờ ({max_wait_seconds}s). Lỗi cuối cùng: {last_err}. Hãy xem log: {LOG_FILE}")

In [ ]:
!PYTHONHASHSEED=42 \
CUBLAS_WORKSPACE_CONFIG=:4096:8 \
lm_eval \
  --model gguf \
  --model_args base_url=http://127.0.0.1:8080 \
  --tasks vietnamese_ssa \
  --apply_chat_template \
  --log_samples \
  --output_path results/lm_eval/gemma_gguf_3shot \
  --metadata '{"technique":"few_shot","language":"vi","n_shot":3}'